<a href="https://colab.research.google.com/github/mdusama6268/halal-stock-analysis/blob/main/notebooks/halal_stock_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HalalStock AI v4 Master — FINAL Event-Safe Version

Run this notebook from top to bottom.  
Do **not** run the old Strict Final Cell after this notebook, because the old cell ignores event risk.

This notebook uses your existing Google Sheet:

**HalalStock AI Database**

Main CEO tab:
- `ceo_summary`

Main decision tabs:
- `final_shortlist`
- `opportunity_hunter`
- `event_risk`
- `daily_report`

Important: This is an educational research and paper-tracking system, not guaranteed investment advice.

In [1]:

# cell 0 ============================================================
# HALALSTOCK AI — NIFTY500 SHARIAH FULL UNIVERSE IMPORTER
# Purpose:
# 1. Scrape latest public Nifty500 Shariah constituents from Screener
# 2. Extract Screener company slugs as NSE symbols
# 3. Update Google Sheet tabs:
#    - shariah_universe
#    - nifty500_shariah_raw
# ============================================================

!pip install -q gspread pandas requests beautifulsoup4 lxml

import re
import time
from datetime import date

import pandas as pd
import requests
from bs4 import BeautifulSoup

import gspread
from google.colab import auth
from google.auth import default

SHEET_NAME = "HalalStock AI Database"
TODAY = str(date.today())

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sheet = gc.open(SHEET_NAME)

BASE_URL = "https://www.screener.in/company/NIFTY500SH/"

def get_ws(tab_name, rows=3000, cols=60):
    try:
        return sheet.worksheet(tab_name)
    except Exception:
        return sheet.add_worksheet(title=tab_name, rows=rows, cols=cols)

def write_tab(tab_name, df):
    ws = get_ws(tab_name)
    ws.clear()
    ws.update([df.columns.tolist()] + df.fillna("").values.tolist())

def clean_number(x):
    try:
        if x is None:
            return ""
        text = str(x).replace(",", "").strip()
        if text == "" or text == "-":
            return ""
        return float(text)
    except Exception:
        return x

def extract_symbol_from_href(href):
    # Screener links look like /company/TCS/consolidated/ or /company/TCS/
    try:
        parts = href.strip("/").split("/")
        if len(parts) >= 2 and parts[0] == "company":
            slug = parts[1].strip().upper()
            if slug and slug not in ["NIFTY500SH"]:
                return slug + ".NS"
        return ""
    except Exception:
        return ""

def parse_page(page_no):
    url = BASE_URL if page_no == 1 else f"{BASE_URL}?page={page_no}"
    print("Fetching:", url)

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "lxml")

    table = soup.find("table")
    if table is None:
        return []

    rows = []
    body_rows = table.find_all("tr")[1:]

    for tr in body_rows:
        cols = [td.get_text(" ", strip=True) for td in tr.find_all("td")]
        if len(cols) < 10:
            continue

        link = tr.find("a", href=True)
        href = link["href"] if link else ""
        symbol = extract_symbol_from_href(href)

        # Expected columns:
        # S.No., Name, CMP Rs., P/E, Mar Cap Rs.Cr., Div Yld %, NP Qtr Rs.Cr.,
        # Qtr Profit Var %, Sales Qtr Rs.Cr., Qtr Sales Var %, ROCE %
        try:
            sno = cols[0].replace(".", "").strip()
            name = cols[1].strip()

            rows.append({
                "sno": sno,
                "symbol": symbol,
                "company_name": name,
                "cmp_rs": clean_number(cols[2]),
                "pe_ratio": clean_number(cols[3]),
                "market_cap_cr": clean_number(cols[4]),
                "div_yield_pct": clean_number(cols[5]),
                "np_qtr_cr": clean_number(cols[6]),
                "qtr_profit_var_pct": clean_number(cols[7]),
                "sales_qtr_cr": clean_number(cols[8]),
                "qtr_sales_var_pct": clean_number(cols[9]),
                "roce_pct": clean_number(cols[10]) if len(cols) > 10 else "",
                "screener_url": "https://www.screener.in" + href if href else "",
                "source": "Screener Nifty500 Shariah",
                "last_updated": TODAY
            })
        except Exception as e:
            print("Row parse error:", e, cols)

    return rows

all_rows = []

for page_no in range(1, 9):
    page_rows = parse_page(page_no)
    all_rows.extend(page_rows)
    time.sleep(1)

raw_df = pd.DataFrame(all_rows)

if raw_df.empty:
    raise Exception("No data fetched. Screener may have blocked the request. Try again later or use Screener export.")

raw_df = raw_df.drop_duplicates(subset=["symbol", "company_name"], keep="first")

# Remove rows without symbols, but keep them in raw export for checking
raw_df = raw_df.sort_values("sno")

universe_df = raw_df[raw_df["symbol"] != ""].copy()

shariah_universe_df = pd.DataFrame({
    "symbol": universe_df["symbol"],
    "company_name": universe_df["company_name"],
    "shariah_status": "Pass",
    "source": "Screener Nifty500 Shariah",
    "last_updated": TODAY,
    "notes": "Universe only; not buy recommendation; verify with Shariah Audit v2"
})

write_tab("nifty500_shariah_raw", raw_df)
write_tab("shariah_universe", shariah_universe_df)

print("DONE")
print("Raw rows fetched:", len(raw_df))
print("Rows written to shariah_universe:", len(shariah_universe_df))
display(shariah_universe_df.head(30))

missing_symbol = raw_df[raw_df["symbol"] == ""]
if not missing_symbol.empty:
    print("Rows needing manual symbol check:")
    display(missing_symbol)


Fetching: https://www.screener.in/company/NIFTY500SH/
Fetching: https://www.screener.in/company/NIFTY500SH/?page=2
Fetching: https://www.screener.in/company/NIFTY500SH/?page=3
Fetching: https://www.screener.in/company/NIFTY500SH/?page=4
Fetching: https://www.screener.in/company/NIFTY500SH/?page=5
Fetching: https://www.screener.in/company/NIFTY500SH/?page=6
Fetching: https://www.screener.in/company/NIFTY500SH/?page=7
Fetching: https://www.screener.in/company/NIFTY500SH/?page=8
DONE
Raw rows fetched: 199
Rows written to shariah_universe: 198


,symbol,company_name,shariah_status,source,last_updated,notes
0,TCS.NS,TCS,Pass,Screener Nifty500 Shariah,2026-07-28,Universe only; not buy recommendation; verify ...
9,DMART.NS,Avenue Super.,Pass,Screener Nifty500 Shariah,2026-07-28,Universe only; not buy recommendation; verify ...
102,WOCKPHARMA.NS,Wockhardt,Pass,Screener Nifty500 Shariah,2026-07-28,Universe only; not buy recommendation; verify ...
104,TATATECH.NS,Tata Technolog.,Pass,Screener Nifty500 Shariah,2026-07-28,Universe only; not buy recommendation; verify ...
105,RRKABEL.NS,R R Kabel,Pass,Screener Nifty500 Shariah,2026-07-28,Universe only; not buy recommendation; verify ...
106,HFCL.NS,HFCL,Pass,Screener Nifty500 Shariah,2026-07-28,Universe only; not buy recommendation; verify ...
107,SHYAMMETL.NS,Shyam Metalics,Pass,Screener Nifty500 Shariah,2026-07-28,Universe only; not buy recommendation; verify ...
108,SAILIFE.NS,Sai Life,Pass,Screener Nifty500 Shariah,2026-07-28,Universe only; not buy recommendation; verify ...
109,MSUMI.NS,Motherson Wiring,Pass,Screener Nifty500 Shariah,2026-07-28,Universe only; not buy recommendation; verify ...
110,ACUTAAS.NS,Acutaas Chemical,Pass,Screener Nifty500 Shariah,2026-07-28,Universe only; not buy recommendation; verify ...


Rows needing manual symbol check:


,sno,symbol,company_name,cmp_rs,pe_ratio,market_cap_cr,div_yield_pct,np_qtr_cr,qtr_profit_var_pct,sales_qtr_cr,qtr_sales_var_pct,roce_pct,screener_url,source,last_updated
25,,,Median: 198 Co.,1103.25,38.05,31773.14,0.5,235.17,17.61,2699.53,14.17,18.88,,Screener Nifty500 Shariah,2026-07-28


In [2]:
# CELL 1: Setup + Google Sheet Connection

!pip install -q gspread pandas yfinance

import math
import time
from datetime import date

import pandas as pd
import yfinance as yf
import gspread

from google.colab import auth
from google.auth import default

SHEET_NAME = "HalalStock AI Database"
TODAY = str(date.today())

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

sheet = gc.open(SHEET_NAME)

print("Connected:", SHEET_NAME)
print("Today:", TODAY)
print("Tabs found:")
for ws in sheet.worksheets():
    print("-", ws.title)

Connected: HalalStock AI Database
Today: 2026-07-28
Tabs found:
- daily_report
- ceo_summary
- event_risk
- opportunity_hunter
- company_checklist
- fundamentals
- final_shortlist
- dashboard_view
- shariah_audit_v2
- shariah_audit
- agent_decisions
- shariah_universe
- prices
- commodity_prices
- agent_scores
- paper_watchlist
- portfolio_tracker
- price_history
- commodity_history
- symbol_validation
- cashflow_quality
- backtest_results
- nifty500_shariah_raw


In [3]:
# CELL 2: Config + Helper Functions

CONFIG = {
    "price_limit": 1000,
    "min_market_cap_cr": 1000,
    "max_market_cap_cr": 50000,
    "min_revenue_growth": 10,
    "min_profit_growth": 10,
    "min_roe": 15,
    "min_roce": 15,
    "max_debt_to_equity": 0.50,
    "max_pe_strict": 35,
    "max_pb_strict": 8,
    "min_checklist_score": 75,
    "shariah_debt_assets_limit": 25,
    "shariah_interest_income_limit": 3,
    "shariah_cash_receivable_assets_limit": 90,
    "tiny_buy_max": "Rs 500-1000 max for learning only",
    "event_block_fall_pct": -5.0,
    "event_high_fall_pct": -8.0,
}

CHECKLIST_WEIGHTS = {
    "business_easy": 4,
    "industry_growing": 5,
    "revenue_growing": 7,
    "profit_growing": 8,
    "margins_stable": 7,
    "ocf_strong": 8,
    "fcf_positive": 8,
    "debt_controlled": 8,
    "roe_roce_strong": 8,
    "management_transparent": 6,
    "auditor_clean": 6,
    "governance_clean": 7,
    "valuation_fair": 8,
    "future_trigger_visible": 6,
    "position_size_controlled": 4,
}

GROWTH_SECTORS = [
    "Technology",
    "Healthcare",
    "Consumer Defensive",
    "Consumer Cyclical",
    "Industrials",
    "Basic Materials",
]

HIGH_RISK_KEYWORDS = [
    "warning",
    "guidance cut",
    "cuts guidance",
    "weak guidance",
    "margin pressure",
    "revenue decline",
    "profit warning",
    "downgrade",
    "target cut",
    "crash",
    "plunge",
    "falls sharply",
    "slumps",
    "fraud",
    "default",
    "auditor resignation",
    "resigns as auditor",
    "sebi",
    "investigation",
    "regulatory action",
    "pledge",
    "promoter pledge",
    "client loss",
    "order cancellation",
    "lower than expected",
    "misses estimates"
]

MEDIUM_RISK_KEYWORDS = [
    "miss",
    "weak",
    "slowdown",
    "pressure",
    "concern",
    "cautious",
    "delay",
    "volatile",
    "under pressure",
    "falls",
    "declines"
]

def get_ws(tab_name, rows=3000, cols=100):
    try:
        return sheet.worksheet(tab_name)
    except Exception:
        return sheet.add_worksheet(title=tab_name, rows=rows, cols=cols)

def read_tab(tab_name):
    try:
        return pd.DataFrame(get_ws(tab_name).get_all_records())
    except Exception:
        return pd.DataFrame()

def write_tab(tab_name, df):
    ws = get_ws(tab_name)
    ws.clear()
    ws.update([df.columns.tolist()] + df.fillna("").values.tolist())

def write_values(tab_name, values):
    ws = get_ws(tab_name)
    ws.clear()
    ws.update(values)

def append_history(tab_name, new_df, unique_cols):
    old_df = read_tab(tab_name)
    if old_df.empty:
        final_df = new_df.copy()
    else:
        final_df = pd.concat([old_df, new_df], ignore_index=True)
        final_df = final_df.drop_duplicates(subset=unique_cols, keep="last")
    write_tab(tab_name, final_df)

def to_num(value):
    try:
        if value is None or value == "":
            return None
        value = float(value)
        if math.isnan(value) or math.isinf(value):
            return None
        return value
    except Exception:
        return None

def yes_no(condition):
    return "Yes" if condition else "No"

def latest_statement_value(statement, row_names):
    try:
        if statement is None or statement.empty:
            return ""
        for name in row_names:
            if name in statement.index:
                values = statement.loc[name].dropna()
                if len(values) > 0:
                    return float(values.iloc[0])
        return ""
    except Exception:
        return ""

def cagr(statement, row_name):
    try:
        if statement is None or statement.empty or row_name not in statement.index:
            return ""
        series = statement.loc[row_name].dropna().sort_index()
        if len(series) < 2:
            return ""
        old = float(series.iloc[0])
        new = float(series.iloc[-1])
        if old <= 0 or new <= 0:
            return ""
        years = max((series.index[-1] - series.index[0]).days / 365.25, 1)
        return round(((new / old) ** (1 / years) - 1) * 100, 2)
    except Exception:
        return ""

def pct(numerator, denominator):
    try:
        if numerator in ["", None] or denominator in ["", None] or denominator == 0:
            return ""
        return round((float(numerator) / float(denominator)) * 100, 2)
    except Exception:
        return ""

def rule_check(value, limit):
    if value == "" or value is None:
        return "Needs Manual Check"
    try:
        return "Pass" if float(value) <= limit else "Fail"
    except Exception:
        return "Needs Manual Check"

def yes_score(value, weight):
    if value == "Yes":
        return weight
    if value == "Check":
        return round(weight * 0.35, 2)
    return 0

def money(value):
    try:
        if value == "" or value is None:
            return ""
        return "Rs " + str(round(float(value), 2))
    except Exception:
        return str(value)

def clean_lower(x):
    return str(x).strip().lower()

print("Config and helper functions loaded.")

Config and helper functions loaded.


In [4]:
# CELL 3: Data Update Agent
# Updates stock prices, average volume, SHARIABEES, Gold ETF, Silver ETF

def get_price_volume(symbol):
    try:
        hist = yf.Ticker(symbol).history(period="30d")
        if hist.empty:
            return "", ""
        close_price = round(float(hist["Close"].dropna().iloc[-1]), 2)
        avg_volume = round(float(hist["Volume"].dropna().tail(20).mean()), 2)
        return close_price, avg_volume
    except Exception:
        return "", ""

shariah_df = read_tab("shariah_universe")

stock_rows = []
for _, row in shariah_df.iterrows():
    symbol = row.get("symbol", "")
    company = row.get("company_name", "")
    if not symbol:
        continue

    price, avg_volume = get_price_volume(symbol)
    stock_rows.append([TODAY, symbol, company, price, avg_volume, "yfinance"])
    time.sleep(0.08)

prices_df = pd.DataFrame(
    stock_rows,
    columns=["date", "symbol", "company_name", "close_price", "avg_volume_20d", "source"]
)

write_tab("prices", prices_df)
append_history("price_history", prices_df, ["date", "symbol"])

commodity_rows = []
for asset, symbol in [
    ["Shariah ETF", "SHARIABEES.NS"],
    ["Gold ETF", "GOLDBEES.NS"],
    ["Silver ETF", "SILVERBEES.NS"],
]:
    price, avg_volume = get_price_volume(symbol)
    commodity_rows.append([TODAY, asset, symbol, price, avg_volume, "yfinance"])

commodity_df = pd.DataFrame(
    commodity_rows,
    columns=["date", "asset", "symbol", "close_price", "avg_volume_20d", "source"]
)

write_tab("commodity_prices", commodity_df)
append_history("commodity_history", commodity_df, ["date", "symbol"])

print("Stock prices updated:", len(prices_df))
display(prices_df.head(20))
print("Commodity prices updated:")
display(commodity_df)

Stock prices updated: 198


,date,symbol,company_name,close_price,avg_volume_20d,source
0,2026-07-28,TCS.NS,TCS,2398.00,4928044.90,yfinance
1,2026-07-28,DMART.NS,Avenue Super.,3846.60,778885.15,yfinance
2,2026-07-28,WOCKPHARMA.NS,Wockhardt,1890.10,602045.60,yfinance
3,2026-07-28,TATATECH.NS,Tata Technolog.,738.60,1259460.25,yfinance
4,2026-07-28,RRKABEL.NS,R R Kabel,2576.90,721773.80,yfinance
5,2026-07-28,HFCL.NS,HFCL,188.39,24411218.30,yfinance
6,2026-07-28,SHYAMMETL.NS,Shyam Metalics,1027.00,623218.05,yfinance
7,2026-07-28,SAILIFE.NS,Sai Life,1300.50,365776.80,yfinance
8,2026-07-28,MSUMI.NS,Motherson Wiring,41.52,9660883.45,yfinance
9,2026-07-28,ACUTAAS.NS,Acutaas Chemical,3297.70,400200.30,yfinance


Commodity prices updated:


,date,asset,symbol,close_price,avg_volume_20d,source
0,2026-07-28,Shariah ETF,SHARIABEES.NS,462.52,7238.30,yfinance
1,2026-07-28,Gold ETF,GOLDBEES.NS,116.83,20136583.60,yfinance
2,2026-07-28,Silver ETF,SILVERBEES.NS,206.20,22174325.95,yfinance


In [5]:
# CELL 4: Symbol Validation Agent

shariah_df = read_tab("shariah_universe")
prices_df = read_tab("prices")

rows = []
for _, row in shariah_df.iterrows():
    symbol = row.get("symbol", "")
    company = row.get("company_name", "")

    match = prices_df[prices_df["symbol"] == symbol]

    if match.empty:
        price = ""
        status = "Missing From Prices"
        action = "Check symbol manually"
    else:
        price = match["close_price"].iloc[0]
        if price == "" or pd.isna(price):
            status = "Price Not Found"
            action = "Check Yahoo symbol or remove temporarily"
        else:
            status = "Valid"
            action = "Use in analysis"

    rows.append([TODAY, symbol, company, price, status, action])

validation_df = pd.DataFrame(
    rows,
    columns=["date", "symbol", "company_name", "close_price", "validation_status", "action_required"]
)

write_tab("symbol_validation", validation_df)

print("Symbol validation completed.")
print("Invalid / missing symbols:")
display(validation_df[validation_df["validation_status"] != "Valid"])

Symbol validation completed.
Invalid / missing symbols:


,date,symbol,company_name,close_price,validation_status,action_required


In [6]:
# CELL 5: Fundamentals Agent
# Updates revenue growth, profit growth, ROE, ROCE, debt, valuation, market cap, sector

validation_df = read_tab("symbol_validation")
valid_df = validation_df[validation_df["validation_status"] == "Valid"]

rows = []
for _, row in valid_df.iterrows():
    symbol = row["symbol"]
    company = row["company_name"]

    try:
        ticker = yf.Ticker(symbol)
        info = ticker.info
        income = ticker.financials
        balance = ticker.balance_sheet

        revenue_growth = cagr(income, "Total Revenue")
        profit_growth = cagr(income, "Net Income")

        roe =  info.get("returnOnEquity", "")
        roe = round(float(roe) * 100, 2) if roe not in ["", None] else ""

        ebit = latest_statement_value(income, ["EBIT"])
        total_assets = latest_statement_value(balance, ["Total Assets"])
        current_liabilities = latest_statement_value(balance, ["Current Liabilities"])

        if ebit != "" and total_assets != "" and current_liabilities != "":
            capital_employed = total_assets - current_liabilities
            roce = round((ebit / capital_employed) * 100, 2) if capital_employed > 0 else ""
        else:
            roce = ""

        debt_to_equity = info.get("debtToEquity", "")
        if debt_to_equity not in ["", None]:
            debt_to_equity = float(debt_to_equity)
            if debt_to_equity > 5:
                debt_to_equity = debt_to_equity / 100
            debt_to_equity = round(debt_to_equity, 2)
        else:
            debt_to_equity = ""

        pe = info.get("trailingPE", "")
        pe = round(float(pe), 2) if pe not in ["", None] else ""

        pb = info.get("priceToBook", "")
        pb = round(float(pb), 2) if pb not in ["", None] else ""

        market_cap = info.get("marketCap", "")
        market_cap = round(float(market_cap) / 10000000, 2) if market_cap not in ["", None] else ""

        sector = info.get("sector", "")

        rows.append([
            symbol, company, revenue_growth, profit_growth, roe, roce,
            debt_to_equity, pe, pb, "", "", market_cap, sector, TODAY, "yfinance_auto_verify"
        ])

        time.sleep(0.12)

    except Exception as e:
        rows.append([
            symbol, company, "", "", "", "", "", "", "", "", "", "", "", TODAY, f"error: {e}"
        ])

fundamentals_df = pd.DataFrame(
    rows,
    columns=[
        "symbol", "company_name", "revenue_growth_5y", "profit_growth_5y",
        "roe", "roce", "debt_to_equity", "pe_ratio", "pb_ratio",
        "promoter_holding", "promoter_pledge", "market_cap", "sector",
        "updated_date", "source"
    ]
)

write_tab("fundamentals", fundamentals_df)

print("Fundamentals updated:", len(fundamentals_df))
display(fundamentals_df.head(20))

Fundamentals updated: 198


,symbol,company_name,revenue_growth_5y,profit_growth_5y,roe,roce,debt_to_equity,pe_ratio,pb_ratio,promoter_holding,promoter_pledge,market_cap,sector,updated_date,source
0,TCS.NS,TCS,5.8,5.3,47.74,54.93,0.1,17.1,7.91,,,,Technology,2026-07-28,yfinance_auto_verify
1,DMART.NS,Avenue Super.,17.13,7.69,,16.41,0.1,81.63,10.23,,,250892.94,Consumer Defensive,2026-07-28,yfinance_auto_verify
2,WOCKPHARMA.NS,Wockhardt,7.82,,4.0,6.38,0.42,144.72,6.21,,,30712.65,Healthcare,2026-07-28,yfinance_auto_verify
3,TATATECH.NS,Tata Technolog.,7.65,-4.32,,15.47,0.24,53.91,7.64,,,29990.99,Technology,2026-07-28,yfinance_auto_verify
4,RRKABEL.NS,R R Kabel,20.08,37.36,20.83,26.51,0.13,59.05,11.32,,,29146.16,Industrials,2026-07-28,yfinance_auto_verify
5,HFCL.NS,HFCL,1.49,1.18,,12.18,0.38,50.24,5.68,,,28828.64,Technology,2026-07-28,yfinance_auto_verify
6,SHYAMMETL.NS,Shyam Metalics,13.73,7.87,,12.95,0.08,25.57,2.54,,,28584.14,Basic Materials,2026-07-28,yfinance_auto_verify
7,SAILIFE.NS,Sai Life,22.35,226.8,15.13,18.10,0.12,78.96,11.07,,,27608.95,Healthcare,2026-07-28,yfinance_auto_verify
8,MSUMI.NS,Motherson Wiring,17.62,8.68,32.39,35.44,0.11,44.17,12.74,,,27534.66,Consumer Cyclical,2026-07-28,yfinance_auto_verify
9,ACUTAAS.NS,Acutaas Chemical,24.34,30.17,,16.51,2.08,91.5,16.33,,,26998.64,Basic Materials,2026-07-28,yfinance_auto_verify


In [7]:
# ============================================================
# HALALSTOCK AI v4 — CELL 5B v3: FUNDAMENTALS REPAIR FIXED
# IMPORTANT:
# Run Cell 5 again first, then run this Cell 5B v3.
#
# What was fixed:
# - Old 5B merged by symbol + company_name.
# - Screener company names and yfinance company names can be different.
# - That caused good yfinance data to be lost.
# - This version merges ONLY by symbol and preserves yfinance data.
# ============================================================

import pandas as pd
from datetime import date

TODAY = str(date.today())

def get_ws(tab_name, rows=3000, cols=100):
    try:
        return sheet.worksheet(tab_name)
    except Exception:
        return sheet.add_worksheet(title=tab_name, rows=rows, cols=cols)

def read_tab(tab_name):
    try:
        return pd.DataFrame(get_ws(tab_name).get_all_records())
    except Exception:
        return pd.DataFrame()

def write_tab(tab_name, df):
    ws = get_ws(tab_name)
    ws.clear()
    ws.update([df.columns.tolist()] + df.fillna("").values.tolist())

def is_blank(value):
    return value is None or value == "" or str(value).lower() in ["nan", "none"]

# -----------------------------
# Load tabs
# -----------------------------

universe = read_tab("shariah_universe")
fundamentals = read_tab("fundamentals")
raw = read_tab("nifty500_shariah_raw")

if universe.empty:
    raise Exception("shariah_universe is empty. Run Universe Importer first.")

if fundamentals.empty:
    raise Exception("fundamentals tab is empty. Run Cell 5 first.")

if raw.empty:
    raise Exception("nifty500_shariah_raw is empty. Run Universe Importer first.")

# -----------------------------
# Required columns
# -----------------------------

fund_cols = [
    "symbol", "company_name", "revenue_growth_5y", "profit_growth_5y",
    "roe", "roce", "debt_to_equity", "pe_ratio", "pb_ratio",
    "promoter_holding", "promoter_pledge", "market_cap", "sector",
    "updated_date", "source"
]

for col in fund_cols:
    if col not in fundamentals.columns:
        fundamentals[col] = ""

for col in [
    "symbol", "company_name", "cmp_rs", "pe_ratio", "market_cap_cr",
    "qtr_profit_var_pct", "qtr_sales_var_pct", "roce_pct"
]:
    if col not in raw.columns:
        raw[col] = ""

for col in ["symbol", "company_name"]:
    if col not in universe.columns:
        raise Exception(f"Missing column in shariah_universe: {col}")

# -----------------------------
# Use symbol as the only merge key
# -----------------------------

base = universe[["symbol", "company_name"]].drop_duplicates(subset=["symbol"], keep="first").copy()
base = base.rename(columns={"company_name": "company_name_universe"})

fund_small = fundamentals[fund_cols].drop_duplicates(subset=["symbol"], keep="first").copy()
fund_small = fund_small.rename(columns={"company_name": "company_name_fund"})

raw_small = raw[
    ["symbol", "company_name", "cmp_rs", "pe_ratio", "market_cap_cr", "qtr_profit_var_pct", "qtr_sales_var_pct", "roce_pct"]
].drop_duplicates(subset=["symbol"], keep="first").copy()
raw_small = raw_small.rename(columns={
    "company_name": "company_name_raw",
    "pe_ratio": "pe_ratio_raw"
})

df = base.merge(fund_small, on="symbol", how="left")
df = df.merge(raw_small, on="symbol", how="left")

rows = []

for _, r in df.iterrows():
    symbol = r.get("symbol", "")

    company = r.get("company_name_fund", "")
    if is_blank(company):
        company = r.get("company_name_universe", "")
    if is_blank(company):
        company = r.get("company_name_raw", "")

    revenue_growth = r.get("revenue_growth_5y", "")
    profit_growth = r.get("profit_growth_5y", "")
    roe = r.get("roe", "")
    roce = r.get("roce", "")
    debt_to_equity = r.get("debt_to_equity", "")
    pe = r.get("pe_ratio", "")
    pb = r.get("pb_ratio", "")
    market_cap = r.get("market_cap", "")
    sector = r.get("sector", "")
    source = r.get("source", "")

    repaired_fields = []
    row_status = "Existing fundamentals"

    if is_blank(source):
        source = "created_from_universe"
        row_status = "Added missing fundamentals row"

    if is_blank(revenue_growth) and not is_blank(r.get("qtr_sales_var_pct")):
        revenue_growth = r.get("qtr_sales_var_pct")
        repaired_fields.append("revenue_growth_from_screener_qtr_sales")

    if is_blank(profit_growth) and not is_blank(r.get("qtr_profit_var_pct")):
        profit_growth = r.get("qtr_profit_var_pct")
        repaired_fields.append("profit_growth_from_screener_qtr_profit")

    if is_blank(roce) and not is_blank(r.get("roce_pct")):
        roce = r.get("roce_pct")
        repaired_fields.append("roce_from_screener")

    if is_blank(pe) and not is_blank(r.get("pe_ratio_raw")):
        pe = r.get("pe_ratio_raw")
        repaired_fields.append("pe_from_screener")

    if is_blank(market_cap) and not is_blank(r.get("market_cap_cr")):
        market_cap = r.get("market_cap_cr")
        repaired_fields.append("market_cap_from_screener")

    # Do not fake ROE, debt, P/B, sector.
    # If they are still missing, the final model should not make a buy signal.

    missing_critical = []
    for field_name, field_value in [
        ("revenue_growth_5y", revenue_growth),
        ("profit_growth_5y", profit_growth),
        ("roe", roe),
        ("roce", roce),
        ("debt_to_equity", debt_to_equity),
        ("pe_ratio", pe),
        ("market_cap", market_cap),
    ]:
        if is_blank(field_value):
            missing_critical.append(field_name)

    if repaired_fields:
        final_source = str(source) + " + screener_fallback"
        data_quality = "Partial - repaired"
    else:
        final_source = source
        data_quality = "Original"

    if row_status == "Added missing fundamentals row":
        data_quality = row_status + " | " + data_quality

    if missing_critical:
        data_quality = data_quality + " | Missing: " + ", ".join(missing_critical)

    rows.append([
        symbol,
        company,
        revenue_growth,
        profit_growth,
        roe,
        roce,
        debt_to_equity,
        pe,
        pb,
        r.get("promoter_holding", ""),
        r.get("promoter_pledge", ""),
        market_cap,
        sector,
        TODAY,
        final_source,
        data_quality,
        "; ".join(repaired_fields) if repaired_fields else ""
    ])

repaired = pd.DataFrame(rows, columns=[
    "symbol", "company_name", "revenue_growth_5y", "profit_growth_5y",
    "roe", "roce", "debt_to_equity", "pe_ratio", "pb_ratio",
    "promoter_holding", "promoter_pledge", "market_cap", "sector",
    "updated_date", "source", "data_quality", "repair_notes"
])

write_tab("fundamentals", repaired)

print("Cell 5B v3 completed.")
print("Total fundamentals rows now:", len(repaired))
print("Rows repaired:", len(repaired[repaired["repair_notes"] != ""]))

print("FSL check:")
display(repaired[repaired["symbol"].astype(str).str.upper() == "FSL.NS"])

print("TCS check, to confirm yfinance data was preserved:")
display(repaired[repaired["symbol"].astype(str).str.upper() == "TCS.NS"])

Cell 5B v3 completed.
Total fundamentals rows now: 198
Rows repaired: 20
FSL check:


,symbol,company_name,revenue_growth_5y,profit_growth_5y,roe,roce,debt_to_equity,pe_ratio,pb_ratio,promoter_holding,promoter_pledge,market_cap,sector,updated_date,source,data_quality,repair_notes
37,FSL.NS,Firstsour.Solu.,17.11,9.49,15.9,17.65,0.67,26.6,4.14,,,18165.22,Technology,2026-07-28,yfinance_auto_verify,Original,


TCS check, to confirm yfinance data was preserved:


,symbol,company_name,revenue_growth_5y,profit_growth_5y,roe,roce,debt_to_equity,pe_ratio,pb_ratio,promoter_holding,promoter_pledge,market_cap,sector,updated_date,source,data_quality,repair_notes
0,TCS.NS,TCS,5.8,5.3,47.74,54.93,0.1,17.1,7.91,,,867544.95,Technology,2026-07-28,yfinance_auto_verify + screener_fallback,Partial - repaired,market_cap_from_screener


In [8]:
# CELL 6: Cash Flow Quality Agent
# Calculates margin, operating cash flow, free cash flow

validation_df = read_tab("symbol_validation")
valid_df = validation_df[validation_df["validation_status"] == "Valid"]

rows = []

for _, row in valid_df.iterrows():
    symbol = row["symbol"]
    company = row["company_name"]

    try:
        ticker = yf.Ticker(symbol)
        income = ticker.financials
        cashflow = ticker.cashflow

        revenue = latest_statement_value(income, ["Total Revenue", "Operating Revenue"])
        net_income = latest_statement_value(income, ["Net Income"])
        ocf = latest_statement_value(cashflow, ["Operating Cash Flow", "Total Cash From Operating Activities"])
        capex = latest_statement_value(cashflow, ["Capital Expenditure", "Capital Expenditures"])

        net_margin_pct = pct(net_income, revenue)

        if ocf != "" and capex != "":
            fcf = float(ocf) + float(capex)
        else:
            fcf = ""

        ocf_strong = "Yes" if ocf != "" and ocf > 0 else "No"
        fcf_positive = "Yes" if fcf != "" and fcf > 0 else "No"
        margins_stable = "Check"

        rows.append([
            TODAY,
            symbol,
            company,
            round(revenue / 10000000, 2) if revenue != "" else "",
            round(net_income / 10000000, 2) if net_income != "" else "",
            net_margin_pct,
            round(ocf / 10000000, 2) if ocf != "" else "",
            round(fcf / 10000000, 2) if fcf != "" else "",
            ocf_strong,
            fcf_positive,
            margins_stable,
            "yfinance_cashflow"
        ])

        time.sleep(0.12)

    except Exception as e:
        rows.append([TODAY, symbol, company, "", "", "", "", "", "Check", "Check", "Check", f"error: {e}"])

cashflow_df = pd.DataFrame(
    rows,
    columns=[
        "date", "symbol", "company_name", "revenue_cr", "net_income_cr", "net_margin_pct",
        "operating_cash_flow_cr", "free_cash_flow_cr", "ocf_strong", "fcf_positive",
        "margins_stable", "source"
    ]
)

write_tab("cashflow_quality", cashflow_df)

print("Cash Flow Quality updated:", len(cashflow_df))
display(cashflow_df.head(20))

Cash Flow Quality updated: 198


,date,symbol,company_name,revenue_cr,net_income_cr,net_margin_pct,operating_cash_flow_cr,free_cash_flow_cr,ocf_strong,fcf_positive,margins_stable,source
0,2026-07-28,TCS.NS,TCS,267021.00,49210.00,18.43,52094.00,47948.00,Yes,Yes,Check,yfinance_cashflow
1,2026-07-28,DMART.NS,Avenue Super.,68684.50,2970.49,4.32,3466.74,-646.61,Yes,No,Check,yfinance_cashflow
2,2026-07-28,WOCKPHARMA.NS,Wockhardt,3317.00,213.00,6.42,390.00,26.00,Yes,Yes,Check,yfinance_cashflow
3,2026-07-28,TATATECH.NS,Tata Technolog.,5505.57,546.59,9.93,775.70,742.11,Yes,Yes,Check,yfinance_cashflow
4,2026-07-28,RRKABEL.NS,R R Kabel,9587.00,492.22,5.13,295.29,4.26,Yes,Yes,Check,yfinance_cashflow
5,2026-07-28,HFCL.NS,HFCL,4949.27,311.74,6.30,-378.13,-723.44,No,No,Check,yfinance_cashflow
6,2026-07-28,SHYAMMETL.NS,Shyam Metalics,18552.21,1070.24,5.77,2023.56,-613.68,Yes,No,Check,yfinance_cashflow
7,2026-07-28,SAILIFE.NS,Sai Life,2192.49,348.91,15.91,509.13,-83.83,Yes,No,Check,yfinance_cashflow
8,2026-07-28,MSUMI.NS,Motherson Wiring,11428.70,625.20,5.47,761.30,559.90,Yes,Yes,Check,yfinance_cashflow
9,2026-07-28,ACUTAAS.NS,Acutaas Chemical,999.97,158.71,15.87,118.34,-76.30,Yes,No,Check,yfinance_cashflow


In [9]:
# CELL 7: Shariah Audit v2 Agent
# Calculates Shariah ratios and blocks failed stocks

validation_df = read_tab("symbol_validation")
valid_df = validation_df[validation_df["validation_status"] == "Valid"]
shariah_df = read_tab("shariah_universe")

valid_df = valid_df.merge(
    shariah_df[["symbol", "shariah_status", "source"]],
    on="symbol",
    how="left"
)

rows = []

for _, row in valid_df.iterrows():
    symbol = row["symbol"]
    company = row["company_name"]
    shariah_status = row.get("shariah_status", "")
    source = row.get("source", "")

    try:
        ticker = yf.Ticker(symbol)
        balance = ticker.balance_sheet
        income = ticker.financials
        info = ticker.info

        total_assets = latest_statement_value(balance, ["Total Assets"])
        interest_bearing_debt = latest_statement_value(
            balance,
            ["Total Debt", "Long Term Debt", "Short Long Term Debt", "Current Debt", "Long Term Debt And Capital Lease Obligation"]
        )
        if interest_bearing_debt == "":
            interest_bearing_debt = info.get("totalDebt", "")

        cash_and_bank = latest_statement_value(
            balance,
            ["Cash And Cash Equivalents", "Cash Cash Equivalents And Short Term Investments", "Cash Financial"]
        )
        receivables = latest_statement_value(balance, ["Accounts Receivable", "Net Receivables", "Receivables"])

        total_income = latest_statement_value(income, ["Total Revenue", "Operating Revenue"])
        interest_income = latest_statement_value(income, ["Interest Income", "Interest Income Non Operating", "Net Interest Income"])

        debt_to_assets_pct = pct(interest_bearing_debt, total_assets)
        interest_income_pct = pct(interest_income, total_income)

        cash_receivable_total = ""
        if cash_and_bank != "" or receivables != "":
            cash_receivable_total = (cash_and_bank if cash_and_bank != "" else 0) + (receivables if receivables != "" else 0)

        cash_receivable_assets_pct = pct(cash_receivable_total, total_assets)

        debt_rule = rule_check(debt_to_assets_pct, CONFIG["shariah_debt_assets_limit"])
        interest_rule = rule_check(interest_income_pct, CONFIG["shariah_interest_income_limit"])
        cash_receivable_rule = rule_check(cash_receivable_assets_pct, CONFIG["shariah_cash_receivable_assets_limit"])

        if str(shariah_status).lower() != "pass":
            final_status = "Fail - Not in Shariah Universe"
        elif debt_rule == "Pass" and interest_rule == "Pass" and cash_receivable_rule == "Pass":
            final_status = "Verified Pass"
        elif debt_rule == "Fail" or interest_rule == "Fail" or cash_receivable_rule == "Fail":
            final_status = "Fail"
        else:
            final_status = "Needs Manual Check"

        rows.append([
            symbol, company,
            round(total_assets / 10000000, 2) if total_assets != "" else "",
            round(float(interest_bearing_debt) / 10000000, 2) if interest_bearing_debt != "" else "",
            round(cash_and_bank / 10000000, 2) if cash_and_bank != "" else "",
            round(receivables / 10000000, 2) if receivables != "" else "",
            round(interest_income / 10000000, 2) if interest_income != "" else "",
            round(total_income / 10000000, 2) if total_income != "" else "",
            debt_to_assets_pct,
            interest_income_pct,
            cash_receivable_assets_pct,
            debt_rule,
            interest_rule,
            cash_receivable_rule,
            final_status,
            source,
            TODAY,
            "Auto Shariah Audit v2 using yfinance; manually verify before real buy"
        ])

        time.sleep(0.12)

    except Exception as e:
        rows.append([
            symbol, company, "", "", "", "", "", "", "", "", "",
            "Needs Manual Check", "Needs Manual Check", "Needs Manual Check",
            "Needs Manual Check", source, TODAY, f"error: {e}"
        ])

audit_df = pd.DataFrame(
    rows,
    columns=[
        "symbol", "company_name", "total_assets", "interest_bearing_debt", "cash_and_bank",
        "receivables", "interest_income", "total_income", "debt_to_assets_pct",
        "interest_income_pct", "cash_receivable_assets_pct", "debt_rule", "interest_rule",
        "cash_receivable_rule", "final_shariah_status", "source", "verified_date", "notes"
    ]
)

write_tab("shariah_audit_v2", audit_df)

print("Shariah Audit v2 updated:", len(audit_df))
display(audit_df.head(20))

Shariah Audit v2 updated: 198


,symbol,company_name,total_assets,interest_bearing_debt,cash_and_bank,receivables,interest_income,total_income,debt_to_assets_pct,interest_income_pct,cash_receivable_assets_pct,debt_rule,interest_rule,cash_receivable_rule,final_shariah_status,source,verified_date,notes
0,TCS.NS,TCS,182372.00,11283.00,6405.00,57630.00,3854.00,267021.00,6.19,1.44,35.11,Pass,Pass,Pass,Verified Pass,Screener Nifty500 Shariah,2026-07-28,Auto Shariah Audit v2 using yfinance; manually...
1,DMART.NS,Avenue Super.,29524.26,2424.99,765.58,149.34,53.98,68684.50,8.21,0.08,3.10,Pass,Pass,Pass,Verified Pass,Screener Nifty500 Shariah,2026-07-28,Auto Shariah Audit v2 using yfinance; manually...
2,WOCKPHARMA.NS,Wockhardt,8615.00,2233.00,217.00,588.00,7.00,3317.00,25.92,0.21,9.34,Fail,Pass,Pass,Fail,Screener Nifty500 Shariah,2026-07-28,Auto Shariah Audit v2 using yfinance; manually...
3,TATATECH.NS,Tata Technolog.,8953.46,937.98,682.34,922.60,35.97,5505.57,10.48,0.65,17.93,Pass,Pass,Pass,Verified Pass,Screener Nifty500 Shariah,2026-07-28,Auto Shariah Audit v2 using yfinance; manually...
4,RRKABEL.NS,R R Kabel,4621.38,337.03,85.53,997.99,1.91,9587.00,7.29,0.02,23.45,Pass,Pass,Pass,Verified Pass,Screener Nifty500 Shariah,2026-07-28,Auto Shariah Audit v2 using yfinance; manually...
5,HFCL.NS,HFCL,8867.58,1896.33,37.67,2212.18,26.06,4949.27,21.38,0.53,25.37,Pass,Pass,Pass,Verified Pass,Screener Nifty500 Shariah,2026-07-28,Auto Shariah Audit v2 using yfinance; manually...
6,SHYAMMETL.NS,Shyam Metalics,20060.84,1005.07,97.12,904.59,125.36,18552.21,5.01,0.68,4.99,Pass,Pass,Pass,Verified Pass,Screener Nifty500 Shariah,2026-07-28,Auto Shariah Audit v2 using yfinance; manually...
7,SAILIFE.NS,Sai Life,3626.14,288.45,81.91,373.62,17.57,2192.49,7.95,0.80,12.56,Pass,Pass,Pass,Verified Pass,Screener Nifty500 Shariah,2026-07-28,Auto Shariah Audit v2 using yfinance; manually...
8,MSUMI.NS,Motherson Wiring,4745.70,233.20,66.20,1884.20,3.40,11428.70,4.91,0.03,41.10,Pass,Pass,Pass,Verified Pass,Screener Nifty500 Shariah,2026-07-28,Auto Shariah Audit v2 using yfinance; manually...
9,ACUTAAS.NS,Acutaas Chemical,1549.26,12.94,185.22,290.49,10.80,999.97,0.84,1.08,30.71,Pass,Pass,Pass,Verified Pass,Screener Nifty500 Shariah,2026-07-28,Auto Shariah Audit v2 using yfinance; manually...


In [10]:
# CELL 8: 15-Point Company Checklist Agent

fundamentals = read_tab("fundamentals")
audit = read_tab("shariah_audit_v2")
cashflow = read_tab("cashflow_quality")

df = audit.merge(fundamentals, on=["symbol", "company_name"], how="left")
df = df.merge(cashflow, on=["symbol", "company_name"], how="left", suffixes=("", "_cf"))

rows = []

for _, r in df.iterrows():
    symbol = r.get("symbol", "")
    company = r.get("company_name", "")
    sector = str(r.get("sector", ""))

    revenue = to_num(r.get("revenue_growth_5y"))
    profit = to_num(r.get("profit_growth_5y"))
    roe = to_num(r.get("roe"))
    roce = to_num(r.get("roce"))
    debt = to_num(r.get("debt_to_equity"))
    pe = to_num(r.get("pe_ratio"))
    pb = to_num(r.get("pb_ratio"))

    shariah = r.get("final_shariah_status", "")

    business_easy = "Yes"
    industry_growing = "Yes" if sector in GROWTH_SECTORS else "Check"
    revenue_growing = yes_no(revenue is not None and revenue >= 7)
    profit_growing = yes_no(profit is not None and profit >= 7)
    margins_stable = r.get("margins_stable", "Check")
    ocf_strong = r.get("ocf_strong", "Check")
    fcf_positive = r.get("fcf_positive", "Check")
    debt_controlled = yes_no(debt is not None and debt <= CONFIG["max_debt_to_equity"])
    roe_roce_strong = yes_no(roe is not None and roce is not None and roe >= CONFIG["min_roe"] and roce >= CONFIG["min_roce"])
    management_transparent = "Check"
    auditor_clean = "Check"
    governance_clean = "Check"
    valuation_fair = yes_no(pe is not None and pe <= 45 and (pb is None or pb <= 10))
    future_trigger_visible = "Check"
    position_size_controlled = "Yes"

    checks = {
        "business_easy": business_easy,
        "industry_growing": industry_growing,
        "revenue_growing": revenue_growing,
        "profit_growing": profit_growing,
        "margins_stable": margins_stable,
        "ocf_strong": ocf_strong,
        "fcf_positive": fcf_positive,
        "debt_controlled": debt_controlled,
        "roe_roce_strong": roe_roce_strong,
        "management_transparent": management_transparent,
        "auditor_clean": auditor_clean,
        "governance_clean": governance_clean,
        "valuation_fair": valuation_fair,
        "future_trigger_visible": future_trigger_visible,
        "position_size_controlled": position_size_controlled,
    }

    checklist_score = sum(yes_score(checks[k], CHECKLIST_WEIGHTS[k]) for k in CHECKLIST_WEIGHTS)
    yes_count = sum(1 for value in checks.values() if value == "Yes")
    check_count = sum(1 for value in checks.values() if value == "Check")

    red_flags = []
    if shariah == "Fail":
        red_flags.append("Shariah fail")
    if debt_controlled == "No":
        red_flags.append("Debt not controlled")
    if ocf_strong == "No":
        red_flags.append("Operating cash flow weak")
    if fcf_positive == "No":
        red_flags.append("Free cash flow negative")
    if valuation_fair == "No":
        red_flags.append("Valuation not fair")

    if shariah == "Fail" or len(red_flags) >= 2:
        final_decision = "Avoid"
        max_allocation = "Rs 0"
    elif shariah == "Verified Pass" and checklist_score >= 85 and len(red_flags) == 0:
        final_decision = "Buy Candidate"
        max_allocation = CONFIG["tiny_buy_max"]
    elif shariah == "Verified Pass" and checklist_score >= 75 and len(red_flags) == 0:
        final_decision = "Tiny Buy Candidate"
        max_allocation = CONFIG["tiny_buy_max"]
    elif shariah == "Verified Pass" and checklist_score >= 60:
        final_decision = "Watch"
        max_allocation = "Rs 0"
    else:
        final_decision = "Avoid / Manual Review"
        max_allocation = "Rs 0"

    reason = f"Score {checklist_score}/100; Yes {yes_count}/15; Check {check_count}/15; Red flags: {', '.join(red_flags) if red_flags else 'None'}"
    exit_trigger = "Exit/review if Shariah fails, debt rises, cash flow weakens, profit declines, auditor/governance issue appears, or valuation becomes extreme."

    rows.append([
        TODAY, symbol, company,
        business_easy, industry_growing, revenue_growing, profit_growing,
        margins_stable, ocf_strong, fcf_positive, debt_controlled, roe_roce_strong,
        management_transparent, auditor_clean, governance_clean, valuation_fair,
        future_trigger_visible, position_size_controlled,
        checklist_score, yes_count, check_count, "; ".join(red_flags),
        final_decision, reason, exit_trigger, max_allocation
    ])

checklist_df = pd.DataFrame(
    rows,
    columns=[
        "date", "symbol", "company_name",
        "business_easy", "industry_growing", "revenue_growing", "profit_growing",
        "margins_stable", "ocf_strong", "fcf_positive", "debt_controlled", "roe_roce_strong",
        "management_transparent", "auditor_clean", "governance_clean", "valuation_fair",
        "future_trigger_visible", "position_size_controlled",
        "checklist_score", "yes_count", "check_count", "red_flags",
        "final_decision", "reason", "exit_trigger", "maximum_allocation"
    ]
)

write_tab("company_checklist", checklist_df)

print("Company checklist updated:", len(checklist_df))
display(checklist_df.sort_values("checklist_score", ascending=False).head(20))

Company checklist updated: 198


,date,symbol,company_name,business_easy,industry_growing,revenue_growing,profit_growing,margins_stable,ocf_strong,fcf_positive,...,future_trigger_visible,position_size_controlled,checklist_score,yes_count,check_count,red_flags,final_decision,reason,exit_trigger,maximum_allocation
20,2026-07-28,CEMPRO.NS,Cemindia Project,Yes,Yes,Yes,Yes,Check,Yes,Yes,...,Check,Yes,79.2,10,5,,Tiny Buy Candidate,Score 79.2/100; Yes 10/15; Check 5/15; Red fla...,"Exit/review if Shariah fails, debt rises, cash...",Rs 500-1000 max for learning only
13,2026-07-28,NBCC.NS,NBCC,Yes,Yes,Yes,Yes,Check,Yes,Yes,...,Check,Yes,79.2,10,5,,Tiny Buy Candidate,Score 79.2/100; Yes 10/15; Check 5/15; Red fla...,"Exit/review if Shariah fails, debt rises, cash...",Rs 500-1000 max for learning only
41,2026-07-28,ECLERX.NS,eClerx Services,Yes,Yes,Yes,Yes,Check,Yes,Yes,...,Check,Yes,79.2,10,5,,Tiny Buy Candidate,Score 79.2/100; Yes 10/15; Check 5/15; Red fla...,"Exit/review if Shariah fails, debt rises, cash...",Rs 500-1000 max for learning only
52,2026-07-28,KPITTECH.NS,KPIT Technologi.,Yes,Yes,Yes,Yes,Check,Yes,Yes,...,Check,Yes,79.2,10,5,,Tiny Buy Candidate,Score 79.2/100; Yes 10/15; Check 5/15; Red fla...,"Exit/review if Shariah fails, debt rises, cash...",Rs 500-1000 max for learning only
180,2026-07-28,AJANTPHARM.NS,Ajanta Pharma,Yes,Yes,Yes,Yes,Check,Yes,Yes,...,Check,Yes,79.2,10,5,,Tiny Buy Candidate,Score 79.2/100; Yes 10/15; Check 5/15; Red fla...,"Exit/review if Shariah fails, debt rises, cash...",Rs 500-1000 max for learning only
126,2026-07-28,LUPIN.NS,Lupin,Yes,Yes,Yes,Yes,Check,Yes,Yes,...,Check,Yes,79.2,10,5,,Tiny Buy Candidate,Score 79.2/100; Yes 10/15; Check 5/15; Red fla...,"Exit/review if Shariah fails, debt rises, cash...",Rs 500-1000 max for learning only
186,2026-07-28,ENDURANCE.NS,Endurance Tech.,Yes,Yes,Yes,Yes,Check,Yes,Yes,...,Check,Yes,79.2,10,5,,Tiny Buy Candidate,Score 79.2/100; Yes 10/15; Check 5/15; Red fla...,"Exit/review if Shariah fails, debt rises, cash...",Rs 500-1000 max for learning only
192,2026-07-28,HEXT.NS,Hexaware Tech.,Yes,Yes,Yes,Yes,Check,Yes,Yes,...,Check,Yes,79.2,10,5,,Tiny Buy Candidate,Score 79.2/100; Yes 10/15; Check 5/15; Red fla...,"Exit/review if Shariah fails, debt rises, cash...",Rs 500-1000 max for learning only
149,2026-07-28,ALKEM.NS,Alkem Lab,Yes,Yes,Yes,Yes,Check,Yes,Yes,...,Check,Yes,79.2,10,5,,Tiny Buy Candidate,Score 79.2/100; Yes 10/15; Check 5/15; Red fla...,"Exit/review if Shariah fails, debt rises, cash...",Rs 500-1000 max for learning only
194,2026-07-28,EMCURE.NS,Emcure Pharma,Yes,Yes,Yes,Yes,Check,Yes,Yes,...,Check,Yes,79.2,10,5,,Tiny Buy Candidate,Score 79.2/100; Yes 10/15; Check 5/15; Red fla...,"Exit/review if Shariah fails, debt rises, cash...",Rs 500-1000 max for learning only


In [11]:
# CELL 9: Auto Event Risk Agent
# This cell is stricter than the previous version.
# Any one-day fall <= -5% OR Medium/High risk news creates action_blocked = Yes.

def price_crash_check(symbol):
    try:
        hist = yf.Ticker(symbol).history(period="7d")
        if hist.empty or len(hist) < 2:
            return None

        close = hist["Close"].dropna()
        if len(close) < 2:
            return None

        last_close = float(close.iloc[-1])
        prev_close = float(close.iloc[-2])

        if prev_close <= 0:
            return None

        one_day_return = ((last_close - prev_close) / prev_close) * 100

        if one_day_return <= CONFIG["event_high_fall_pct"]:
            return {
                "event_type": "Price Crash",
                "severity": "High",
                "source": "yfinance price action",
                "summary": f"Stock fell {round(one_day_return, 2)}% in one trading session.",
                "blocked": "Yes",
                "block_until": "Manual review after reason is known",
                "notes": ""
            }

        if one_day_return <= CONFIG["event_block_fall_pct"]:
            return {
                "event_type": "Sharp Fall",
                "severity": "Medium",
                "source": "yfinance price action",
                "summary": f"Stock fell {round(one_day_return, 2)}% in one trading session.",
                "blocked": "Yes",
                "block_until": "Manual review after next result/update",
                "notes": ""
            }

        return None

    except Exception:
        return None

def scan_news(symbol):
    try:
        ticker = yf.Ticker(symbol)
        news_items = ticker.news

        if not news_items:
            return None

        for item in news_items[:10]:
            title_raw = str(item.get("title", ""))
            title = title_raw.lower()
            publisher = str(item.get("publisher", ""))
            link = str(item.get("link", ""))

            for keyword in HIGH_RISK_KEYWORDS:
                if keyword in title:
                    return {
                        "event_type": "News Risk",
                        "severity": "High",
                        "source": publisher,
                        "summary": title_raw,
                        "blocked": "Yes",
                        "block_until": "Manual review after next result/update",
                        "notes": link
                    }

            for keyword in MEDIUM_RISK_KEYWORDS:
                if keyword in title:
                    return {
                        "event_type": "News Risk",
                        "severity": "Medium",
                        "source": publisher,
                        "summary": title_raw,
                        "blocked": "Yes",
                        "block_until": "Manual review before buy",
                        "notes": link
                    }

        return None

    except Exception:
        return None

existing_event_df = read_tab("event_risk")
validation_df = read_tab("symbol_validation")
valid_symbols = validation_df[validation_df["validation_status"] == "Valid"]

auto_rows = []

for _, row in valid_symbols.iterrows():
    symbol = row.get("symbol", "")
    company = row.get("company_name", "")

    crash_event = price_crash_check(symbol)
    news_event = scan_news(symbol)

    selected_event = None

    if crash_event and crash_event["severity"] == "High":
        selected_event = crash_event
    elif news_event and news_event["severity"] == "High":
        selected_event = news_event
    elif crash_event:
        selected_event = crash_event
    elif news_event:
        selected_event = news_event

    if selected_event:
        auto_rows.append([
            TODAY,
            symbol,
            company,
            selected_event["event_type"],
            selected_event["severity"],
            selected_event["source"],
            selected_event["summary"],
            selected_event["blocked"],
            selected_event["block_until"],
            selected_event.get("notes", "")
        ])

auto_event_df = pd.DataFrame(auto_rows, columns=[
    "date",
    "symbol",
    "company_name",
    "event_type",
    "severity",
    "source",
    "event_summary",
    "action_blocked",
    "block_until",
    "notes"
])

# Preserve manual event-risk entries and combine with auto events.
if existing_event_df.empty:
    final_event_df = auto_event_df
else:
    needed_cols = auto_event_df.columns.tolist()
    for col in needed_cols:
        if col not in existing_event_df.columns:
            existing_event_df[col] = ""
    existing_event_df = existing_event_df[needed_cols]
    final_event_df = pd.concat([existing_event_df, auto_event_df], ignore_index=True)

if not final_event_df.empty:
    final_event_df["block_rank"] = final_event_df["action_blocked"].apply(lambda x: 1 if clean_lower(x) == "yes" else 0)
    final_event_df["severity_rank"] = final_event_df["severity"].apply(lambda x: 2 if clean_lower(x) == "high" else (1 if clean_lower(x) == "medium" else 0))
    final_event_df = final_event_df.sort_values(["symbol", "block_rank", "severity_rank"], ascending=[True, False, False])
    final_event_df = final_event_df.drop_duplicates(subset=["symbol"], keep="first")
    final_event_df = final_event_df.drop(columns=["block_rank", "severity_rank"])

write_tab("event_risk", final_event_df)

print("Auto Event Risk Agent completed.")
print("Events found:", len(final_event_df))
display(final_event_df)

Auto Event Risk Agent completed.
Events found: 6


,date,symbol,company_name,event_type,severity,source,event_summary,action_blocked,block_until,notes
3,2026-07-28,GRAVITA.NS,Gravita India,Price Crash,High,yfinance price action,Stock fell -8.4% in one trading session.,Yes,Manual review after reason is known,
4,2026-07-28,HINDUNILVR.NS,Hind. Unilever,Sharp Fall,Medium,yfinance price action,Stock fell -6.99% in one trading session.,Yes,Manual review after next result/update,
0,2026-07-07,HSCL.NS,Himadri Special,Sharp Fall,Medium,yfinance price action,Stock fell -5.93% in one trading session.,Yes,Manual review after next result/update,
1,2026-07-01,KPITTECH.NS,KPIT Technologies Ltd,Sharp Fall,Medium,yfinance price action,Stock fell -5.71% in one trading session.,Yes,Manual review after next result/update,
5,2026-07-28,SUZLON.NS,Suzlon Energy,Price Crash,High,yfinance price action,Stock fell -9.65% in one trading session.,Yes,Manual review after reason is known,
2,2026-07-07,TRENT.NS,Trent,Price Crash,High,yfinance price action,Stock fell -12.44% in one trading session.,Yes,Manual review after reason is known,


In [12]:
# CELL 10: Event Risk + Strict Opportunity Hunter + CEO Summary
# This is the only final decision cell. Do not run old strict final cells after this.

def valuation_status(pe, pb):
    pe = to_num(pe)
    pb = to_num(pb)

    if pe is None:
        return "Needs Check"
    if pe <= 25 and (pb is None or pb <= 5):
        return "Fair"
    if pe <= CONFIG["max_pe_strict"] and (pb is None or pb <= CONFIG["max_pb_strict"]):
        return "Reasonable"
    if pe <= 45:
        return "Expensive"
    return "Very Expensive"

def get_event_risk_map():
    event_df = read_tab("event_risk")
    event_map = {}

    if event_df.empty:
        return event_map

    for _, r in event_df.iterrows():
        symbol = str(r.get("symbol", "")).strip()
        blocked = clean_lower(r.get("action_blocked", ""))
        severity = clean_lower(r.get("severity", ""))

        # World-class strict rule:
        # If action_blocked is Yes OR severity is Medium/High, block from buy.
        if symbol and (blocked == "yes" or severity in ["medium", "high"]):
            event_map[symbol] = {
                "event_type": r.get("event_type", ""),
                "severity": r.get("severity", ""),
                "summary": r.get("event_summary", ""),
                "block_until": r.get("block_until", ""),
            }

    return event_map

def risk_status(row):
    debt = to_num(row.get("debt_to_equity"))
    pe = to_num(row.get("pe_ratio"))
    roe = to_num(row.get("roe"))
    roce = to_num(row.get("roce"))
    ocf = str(row.get("ocf_strong", "Check"))
    fcf = str(row.get("fcf_positive", "Check"))
    red_flags = str(row.get("red_flags", ""))

    points = 0

    if debt is not None and debt > CONFIG["max_debt_to_equity"]:
        points += 2
    if pe is not None and pe > 45:
        points += 2
    elif pe is not None and pe > CONFIG["max_pe_strict"]:
        points += 1
    if roe is not None and roe < 12:
        points += 1
    if roce is not None and roce < 12:
        points += 1
    if ocf == "No":
        points += 2
    if fcf == "No":
        points += 2
    if red_flags and red_flags.lower() not in ["none", "nan", ""]:
        points += 2

    if points >= 4:
        return "High Risk"
    if points >= 2:
        return "Medium Risk"
    return "Low Risk"

def hard_buy_lock(row):
    failures = []

    price = to_num(row.get("close_price"))
    market_cap = to_num(row.get("market_cap"))
    revenue = to_num(row.get("revenue_growth_5y"))
    profit = to_num(row.get("profit_growth_5y"))
    roe = to_num(row.get("roe"))
    roce = to_num(row.get("roce"))
    debt = to_num(row.get("debt_to_equity"))
    pe = to_num(row.get("pe_ratio"))
    pb = to_num(row.get("pb_ratio"))
    checklist_score = to_num(row.get("checklist_score"))
    shariah = str(row.get("final_shariah_status", ""))
    ocf = str(row.get("ocf_strong", "Check"))
    fcf = str(row.get("fcf_positive", "Check"))
    red_flags = str(row.get("red_flags", ""))

    if shariah != "Verified Pass":
        failures.append("Shariah not verified")
    if price is None or price > CONFIG["price_limit"]:
        failures.append("Share price above opportunity filter")
    if market_cap is None or market_cap < CONFIG["min_market_cap_cr"] or market_cap > CONFIG["max_market_cap_cr"]:
        failures.append("Market cap outside 1000-50000 Cr")
    if revenue is None or revenue < CONFIG["min_revenue_growth"]:
        failures.append("Revenue growth below 10%")
    if profit is None or profit < CONFIG["min_profit_growth"]:
        failures.append("Profit growth below 10%")
    if roe is None or roe < CONFIG["min_roe"]:
        failures.append("ROE below 15%")
    if roce is None or roce < CONFIG["min_roce"]:
        failures.append("ROCE below 15%")
    if debt is None or debt > CONFIG["max_debt_to_equity"]:
        failures.append("Debt not controlled")
    if pe is None or pe > CONFIG["max_pe_strict"]:
        failures.append("P/E above strict limit")
    if pb is not None and pb > CONFIG["max_pb_strict"]:
        failures.append("P/B above strict limit")
    if checklist_score is None or checklist_score < CONFIG["min_checklist_score"]:
        failures.append("Checklist score below 75")
    if ocf == "No":
        failures.append("Operating cash flow weak")
    if fcf == "No":
        failures.append("Free cash flow negative")
    if red_flags and red_flags.lower() not in ["none", "nan", ""]:
        failures.append("Red flags present")

    return failures

def opportunity_score(row):
    score = 0

    price = to_num(row.get("close_price"))
    market_cap = to_num(row.get("market_cap"))
    revenue = to_num(row.get("revenue_growth_5y"))
    profit = to_num(row.get("profit_growth_5y"))
    roe = to_num(row.get("roe"))
    roce = to_num(row.get("roce"))
    debt = to_num(row.get("debt_to_equity"))
    pe = to_num(row.get("pe_ratio"))
    pb = to_num(row.get("pb_ratio"))
    checklist = to_num(row.get("checklist_score"))
    shariah = str(row.get("final_shariah_status", ""))
    ocf = str(row.get("ocf_strong", "Check"))
    fcf = str(row.get("fcf_positive", "Check"))

    if shariah == "Verified Pass":
        score += 20
    if price is not None and price <= CONFIG["price_limit"]:
        score += 8
    if market_cap is not None and CONFIG["min_market_cap_cr"] <= market_cap <= CONFIG["max_market_cap_cr"]:
        score += 8
    if revenue is not None:
        score += min(max(revenue, 0), 25) * 0.4
    if profit is not None:
        score += min(max(profit, 0), 25) * 0.5
    if roe is not None and roe >= CONFIG["min_roe"]:
        score += 8
    if roce is not None and roce >= CONFIG["min_roce"]:
        score += 8
    if debt is not None and debt <= CONFIG["max_debt_to_equity"]:
        score += 8
    if pe is not None and pe <= CONFIG["max_pe_strict"]:
        score += 6
    if pb is None or pb <= CONFIG["max_pb_strict"]:
        score += 4
    if ocf == "Yes":
        score += 5
    elif ocf == "Check":
        score += 2
    if fcf == "Yes":
        score += 5
    elif fcf == "Check":
        score += 2
    if checklist is not None:
        score += min(checklist * 0.10, 10)

    return round(score, 2)

prices = read_tab("prices")
fundamentals = read_tab("fundamentals")
audit = read_tab("shariah_audit_v2")
checklist = read_tab("company_checklist")
cashflow = read_tab("cashflow_quality")
event_map = get_event_risk_map()

df = audit.merge(fundamentals, on=["symbol", "company_name"], how="left")
df = df.merge(prices[["symbol", "close_price", "avg_volume_20d"]], on="symbol", how="left")
df = df.merge(
    checklist[["symbol", "checklist_score", "yes_count", "check_count", "red_flags"]],
    on="symbol",
    how="left"
)
df = df.merge(
    cashflow[["symbol", "ocf_strong", "fcf_positive", "net_margin_pct"]],
    on="symbol",
    how="left"
)

rows = []

for _, row in df.iterrows():
    symbol = str(row.get("symbol", "")).strip()

    failures = hard_buy_lock(row)
    valuation = valuation_status(row.get("pe_ratio"), row.get("pb_ratio"))
    risk = risk_status(row)
    score = opportunity_score(row)

    event = event_map.get(symbol)
    event_blocked = "Yes" if event else "No"
    event_summary = event["summary"] if event else ""

    if event:
        decision = "Watch / Event Risk Block"
        allocation = "Rs 0"
        reason = f"Blocked by live event risk: {event.get('event_type', '')}"
        failures.append("Live event risk block")
    elif len(failures) == 0 and risk == "Low Risk" and score >= 80:
        decision = "Tiny Buy Candidate"
        allocation = CONFIG["tiny_buy_max"]
        reason = "Passed strict buy locks"
    elif str(row.get("final_shariah_status", "")) != "Verified Pass":
        decision = "Avoid"
        allocation = "Rs 0"
        reason = "Failed Shariah gate"
    elif risk == "High Risk":
        decision = "Avoid"
        allocation = "Rs 0"
        reason = "High risk"
    elif score >= 65:
        decision = "Watch"
        allocation = "Rs 0"
        reason = "Potential opportunity but strict buy locks failed"
    else:
        decision = "Avoid / Wait"
        allocation = "Rs 0"
        reason = "Opportunity score too low"

    rows.append([
        TODAY,
        symbol,
        row.get("company_name", ""),
        row.get("close_price", ""),
        row.get("market_cap", ""),
        row.get("sector", ""),
        row.get("final_shariah_status", ""),
        row.get("revenue_growth_5y", ""),
        row.get("profit_growth_5y", ""),
        row.get("roe", ""),
        row.get("roce", ""),
        row.get("debt_to_equity", ""),
        row.get("pe_ratio", ""),
        row.get("pb_ratio", ""),
        valuation,
        row.get("ocf_strong", "Check"),
        row.get("fcf_positive", "Check"),
        row.get("checklist_score", ""),
        risk,
        score,
        event_blocked,
        event_summary,
        decision,
        reason,
        "; ".join(failures) if failures else "None",
        allocation
    ])

opportunity_df = pd.DataFrame(rows, columns=[
    "date",
    "symbol",
    "company_name",
    "share_price",
    "market_cap",
    "sector",
    "shariah_status",
    "revenue_growth",
    "profit_growth",
    "roe",
    "roce",
    "debt_to_equity",
    "pe_ratio",
    "pb_ratio",
    "valuation_status",
    "ocf_strong",
    "fcf_positive",
    "checklist_score",
    "risk_status",
    "opportunity_score",
    "event_blocked",
    "event_summary",
    "final_decision",
    "reason",
    "failed_rules",
    "maximum_allocation"
])

# Custom sort: Tiny candidates first, then event blocks, then watch, then avoid.
decision_order = {
    "Tiny Buy Candidate": 1,
    "Watch / Event Risk Block": 2,
    "Watch": 3,
    "Avoid / Wait": 4,
    "Avoid": 5,
}
opportunity_df["decision_sort"] = opportunity_df["final_decision"].map(decision_order).fillna(9)
opportunity_df = opportunity_df.sort_values(["decision_sort", "opportunity_score"], ascending=[True, False])
opportunity_df = opportunity_df.drop(columns=["decision_sort"])

write_tab("opportunity_hunter", opportunity_df)

final_shortlist = opportunity_df[
    opportunity_df["final_decision"] == "Tiny Buy Candidate"
].sort_values("opportunity_score", ascending=False)

write_tab("final_shortlist", final_shortlist)

blocked = opportunity_df[opportunity_df["final_decision"] == "Watch / Event Risk Block"]
tiny = final_shortlist.head(10)
watch = opportunity_df[opportunity_df["final_decision"] == "Watch"].head(10)
avoid = opportunity_df[opportunity_df["final_decision"].isin(["Avoid", "Avoid / Wait"])].head(10)

def make_summary(data):
    if data.empty:
        return "None"
    return "; ".join(
        f"{r['company_name']} ({r['symbol']})"
        for _, r in data.head(5).iterrows()
    )

report_df = pd.DataFrame([
    [TODAY, "Event Risk + Strict FINAL", "Tiny Buy Candidates", make_summary(tiny), "Manual verify before even Rs 500-1000"],
    [TODAY, "Event Risk + Strict FINAL", "Event Risk Blocked", make_summary(blocked), "Do not buy until event clears"],
    [TODAY, "Event Risk + Strict FINAL", "Watchlist", make_summary(watch), "Track only"],
    [TODAY, "Event Risk + Strict FINAL", "Avoid / Wait", make_summary(avoid), "Do not buy"],
    [TODAY, "Event Risk + Strict FINAL", "Final Action", "Event risk gate is active. final_shortlist excludes blocked stocks.", "Research only"]
], columns=["date", "report_type", "section", "summary", "action_required"])

write_tab("daily_report", report_df)

# CEO summary
if len(final_shortlist) > 0:
    action = "TINY TEST POSSIBLE"
    top = final_shortlist.iloc[0]
    top_company = top.get("company_name", "")
    top_symbol = top.get("symbol", "")
    top_score = top.get("opportunity_score", "")
    top_price = top.get("share_price", "")
    top_risk = top.get("risk_status", "")
    top_valuation = top.get("valuation_status", "")
    top_allocation = top.get("maximum_allocation", "")
else:
    action = "NO BUY TODAY"
    top_company = "None"
    top_symbol = ""
    top_score = ""
    top_price = ""
    top_risk = ""
    top_valuation = ""
    top_allocation = "Rs 0"

commodity = read_tab("commodity_prices")

ceo_values = [
    ["HALALSTOCK AI — CEO SUMMARY", "", "", "", ""],
    ["Updated", TODAY, "", "", ""],
    ["Final Action", action, "", "", ""],
    ["Top Candidate", top_company, top_symbol, top_score, ""],
    ["Top Price", top_price, "", "", ""],
    ["Top Risk", top_risk, "", "", ""],
    ["Top Valuation", top_valuation, "", "", ""],
    ["Max Allocation", top_allocation, "", "", ""],
    ["Tiny Buy Count", len(final_shortlist), "", "", ""],
    ["Event Blocked", make_summary(blocked), "", "", ""],
    ["", "", "", "", ""],
    ["TOP 5 FINAL SHORTLIST", "", "", "", ""],
    ["Rank", "Symbol", "Company", "Score", "Max Allocation"],
]

if len(final_shortlist) > 0:
    for i, (_, r) in enumerate(final_shortlist.head(5).iterrows(), start=1):
        ceo_values.append([i, r.get("symbol", ""), r.get("company_name", ""), r.get("opportunity_score", ""), r.get("maximum_allocation", "")])
else:
    ceo_values.append(["-", "None", "No strict candidate today", "", ""])

ceo_values += [
    ["", "", "", "", ""],
    ["TOP 5 EVENT BLOCKED", "", "", "", ""],
    ["Rank", "Symbol", "Company", "Event", "Action"],
]

if not blocked.empty:
    for i, (_, r) in enumerate(blocked.head(5).iterrows(), start=1):
        ceo_values.append([i, r.get("symbol", ""), r.get("company_name", ""), r.get("event_summary", ""), "No Buy"])
else:
    ceo_values.append(["-", "None", "No event block", "", ""])

ceo_values += [
    ["", "", "", "", ""],
    ["TOP 5 WATCHLIST", "", "", "", ""],
    ["Rank", "Symbol", "Company", "Score", "Reason"],
]

if not watch.empty:
    for i, (_, r) in enumerate(watch.head(5).iterrows(), start=1):
        ceo_values.append([i, r.get("symbol", ""), r.get("company_name", ""), r.get("opportunity_score", ""), r.get("reason", "")])
else:
    ceo_values.append(["-", "None", "No watchlist data", "", ""])

ceo_values += [
    ["", "", "", "", ""],
    ["MARKET SNAPSHOT", "", "", "", ""],
    ["Asset", "Symbol", "Price", "Source", ""],
]

if not commodity.empty:
    for _, r in commodity.iterrows():
        ceo_values.append([r.get("asset", ""), r.get("symbol", ""), money(r.get("close_price", "")), r.get("source", ""), ""])
else:
    ceo_values.append(["No commodity data", "", "", "", ""])

ceo_values += [
    ["", "", "", "", ""],
    ["RULE", "Only final_shortlist can be considered.", "", "", ""],
    ["RULE", "Event blocked stocks = Rs 0 allocation.", "", "", ""],
    ["RULE", "If final_shortlist is empty, decision is NO BUY TODAY.", "", "", ""],
    ["RULE", "Direct shares are tiny learning tests; core investment stays Tata Ethical / SHARIABEES.", "", "", ""],
]

write_values("ceo_summary", ceo_values)

print("Event Risk + Strict FINAL completed.")
print("Final shortlist count:", len(final_shortlist))
print("Event blocked count:", len(blocked))
display(report_df)

/tmp/ipykernel_552/4222896681.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ws.update([df.columns.tolist()] + df.fillna("").values.tolist())


Event Risk + Strict FINAL completed.
Final shortlist count: 1
Event blocked count: 6


,date,report_type,section,summary,action_required
0,2026-07-28,Event Risk + Strict FINAL,Tiny Buy Candidates,Hexaware Tech. (HEXT.NS),Manual verify before even Rs 500-1000
1,2026-07-28,Event Risk + Strict FINAL,Event Risk Blocked,KPIT Technologi. (KPITTECH.NS); Suzlon Energy ...,Do not buy until event clears
2,2026-07-28,Event Risk + Strict FINAL,Watchlist,NBCC (NBCC.NS); Usha Martin (USHAMART.NS); The...,Track only
3,2026-07-28,Event Risk + Strict FINAL,Avoid / Wait,O N G C (ONGC.NS); Jubilant Ingrev. (JUBLINGRE...,Do not buy
4,2026-07-28,Event Risk + Strict FINAL,Final Action,Event risk gate is active. final_shortlist exc...,Research only


In [13]:
# CELL 11: Paper Watchlist Agent
# Saves today's Tiny Buy Candidates for future backtesting

opportunity = read_tab("opportunity_hunter")
prices = read_tab("prices")
commodity = read_tab("commodity_prices")

candidates = opportunity[opportunity["final_decision"] == "Tiny Buy Candidate"].head(10)

benchmark_symbol = "SHARIABEES.NS"
benchmark_match = commodity[commodity["symbol"] == benchmark_symbol]
benchmark_price = benchmark_match["close_price"].iloc[0] if not benchmark_match.empty else ""

existing = read_tab("paper_watchlist")

rows = []

for _, r in candidates.iterrows():
    symbol = r["symbol"]
    company = r["company_name"]

    price_match = prices[prices["symbol"] == symbol]
    entry_price = price_match["close_price"].iloc[0] if not price_match.empty else ""

    duplicate = False
    if not existing.empty:
        d = existing[
            (existing["recommendation_date"].astype(str) == TODAY)
            & (existing["symbol"].astype(str) == symbol)
        ]
        duplicate = not d.empty

    if not duplicate:
        rows.append([
            TODAY,
            symbol,
            company,
            r.get("opportunity_score", ""),
            r.get("final_decision", ""),
            entry_price,
            benchmark_symbol,
            benchmark_price,
            "Active",
            "",
            "",
            "",
            "Pending",
            "Auto-added by HalalStock AI v4 Event-Safe Model"
        ])

new_watchlist = pd.DataFrame(
    rows,
    columns=[
        "recommendation_date", "symbol", "company_name", "score", "rating",
        "entry_price", "benchmark_symbol", "benchmark_price", "status",
        "return_1m", "return_3m", "return_6m", "result", "notes"
    ]
)

if existing.empty:
    final_watchlist = new_watchlist
else:
    final_watchlist = pd.concat([existing, new_watchlist], ignore_index=True)

write_tab("paper_watchlist", final_watchlist)

print("Paper watchlist new rows:", len(new_watchlist))
display(new_watchlist)

Paper watchlist new rows: 1


,recommendation_date,symbol,company_name,score,rating,entry_price,benchmark_symbol,benchmark_price,status,return_1m,return_3m,return_6m,result,notes
0,2026-07-28,HEXT.NS,Hexaware Tech.,101.14,Tiny Buy Candidate,593.15,SHARIABEES.NS,462.52,Active,,,,Pending,Auto-added by HalalStock AI v4 Event-Safe Model


In [14]:
# CELL 12: Backtest Agent
# Updates current return for active paper watchlist against SHARIABEES

watchlist = read_tab("paper_watchlist")
prices = read_tab("prices")
commodity = read_tab("commodity_prices")

if watchlist.empty:
    print("paper_watchlist is empty. Nothing to backtest.")
else:
    rows = []

    for _, r in watchlist.iterrows():
        symbol = r.get("symbol", "")
        entry_price = to_num(r.get("entry_price"))
        benchmark_symbol = r.get("benchmark_symbol", "SHARIABEES.NS")
        benchmark_entry = to_num(r.get("benchmark_price"))

        price_match = prices[prices["symbol"] == symbol]
        current_price = to_num(price_match["close_price"].iloc[0]) if not price_match.empty else None

        bench_match = commodity[commodity["symbol"] == benchmark_symbol]
        current_benchmark = to_num(bench_match["close_price"].iloc[0]) if not bench_match.empty else None

        stock_return = ""
        benchmark_return = ""
        beat_benchmark = ""

        if entry_price and current_price:
            stock_return = round(((current_price - entry_price) / entry_price) * 100, 2)

        if benchmark_entry and current_benchmark:
            benchmark_return = round(((current_benchmark - benchmark_entry) / benchmark_entry) * 100, 2)

        if stock_return != "" and benchmark_return != "":
            beat_benchmark = "Yes" if stock_return > benchmark_return else "No"

        rows.append([
            TODAY,
            r.get("recommendation_date", ""),
            symbol,
            r.get("company_name", ""),
            entry_price,
            current_price if current_price is not None else "",
            stock_return,
            benchmark_symbol,
            benchmark_entry if benchmark_entry is not None else "",
            current_benchmark if current_benchmark is not None else "",
            benchmark_return,
            beat_benchmark,
            r.get("rating", ""),
            r.get("status", "")
        ])

    backtest_df = pd.DataFrame(
        rows,
        columns=[
            "run_date", "recommendation_date", "symbol", "company_name",
            "entry_price", "current_price", "stock_return_pct",
            "benchmark_symbol", "benchmark_entry_price", "benchmark_current_price",
            "benchmark_return_pct", "beat_benchmark", "rating", "status"
        ]
    )

    write_tab("backtest_results", backtest_df)

    print("Backtest updated:", len(backtest_df))
    display(backtest_df)

Backtest updated: 25


,run_date,recommendation_date,symbol,company_name,entry_price,current_price,stock_return_pct,benchmark_symbol,benchmark_entry_price,benchmark_current_price,benchmark_return_pct,beat_benchmark,rating,status
0,2026-07-28,2026-06-30,HCLTECH.NS,HCL Technologies Ltd,1071.80,1318.60,23.03,SHARIABEES.NS,443.07,462.52,4.39,Yes,Strong Watchlist,Active
1,2026-07-28,2026-06-30,INFY.NS,Infosys Ltd,1000.40,1105.70,10.53,SHARIABEES.NS,443.07,462.52,4.39,Yes,Strong Watchlist,Active
2,2026-07-28,2026-06-30,TCS.NS,Tata Consultancy Services Ltd,2031.50,2398.00,18.04,SHARIABEES.NS,443.07,462.52,4.39,Yes,Strong Watchlist,Active
3,2026-07-28,2026-06-30,CUMMINSIND.NS,Cummins India Ltd,5659.50,5455.50,-3.60,SHARIABEES.NS,443.07,462.52,4.39,No,Strong Watchlist,Active
4,2026-07-28,2026-06-30,TRENT.NS,Trent Ltd,3282.60,2930.70,-10.72,SHARIABEES.NS,443.07,462.52,4.39,No,Strong Watchlist,Active
5,2026-07-28,2026-07-01,HCLTECH.NS,HCL Technologies Ltd,1034.20,1318.60,27.50,SHARIABEES.NS,442.16,462.52,4.60,Yes,Strong Watchlist,Active
6,2026-07-28,2026-07-01,INFY.NS,Infosys Ltd,985.30,1105.70,12.22,SHARIABEES.NS,442.16,462.52,4.60,Yes,Strong Watchlist,Active
7,2026-07-28,2026-07-01,TCS.NS,Tata Consultancy Services Ltd,1982.60,2398.00,20.95,SHARIABEES.NS,442.16,462.52,4.60,Yes,Strong Watchlist,Active
8,2026-07-28,2026-07-01,CUMMINSIND.NS,Cummins India Ltd,5663.00,5455.50,-3.66,SHARIABEES.NS,442.16,462.52,4.60,No,Strong Watchlist,Active
9,2026-07-28,2026-07-01,TRENT.NS,Trent Ltd,3290.30,2930.70,-10.93,SHARIABEES.NS,442.16,462.52,4.60,No,Strong Watchlist,Active
